In [1]:
import kagglehub
import pandas as pd
import os

"""
#To set path to Desktop in macOS/Linux
os.environ["KAGGLEHUB_CACHE"] = os.path.expanduser("~/Desktop/kaggle_data")

#To set path to Desktop in Windows
os.environ["KAGGLEHUB_CACHE"] = os.path.join(os.path.expanduser("~"), "Desktop", "kaggle_data")

Or you can leave as it is and let it download in default folder.
"""
os.environ["KAGGLEHUB_CACHE"] = os.path.expanduser("~/Desktop/kaggle_data")
path = kagglehub.dataset_download("asaniczka/tmdb-movies-dataset-2023-930k-movies")
print("Dataset downloaded to:", path)

csv_file = [f for f in os.listdir(path) if f.endswith(".csv")][0]
raw_path = os.path.join(path, csv_file)

movies = pd.read_csv(raw_path, usecols=[
    "title", "release_date", "vote_average", "vote_count",
    "popularity", "overview", "genres", "keywords", "runtime", "adult"
])

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 248M/248M [00:15<00:00, 17.2MB/s] 

Extracting files...


Dataset downloaded to: /Users/sujay/Desktop/kaggle_data/datasets/asaniczka/tmdb-movies-dataset-2023-930k-movies/versions/997


In [2]:
print(movies.info())
print(movies.head())

<class 'pandas.DataFrame'>
RangeIndex: 1458474 entries, 0 to 1458473
Data columns (total 10 columns):
 #   Column        Non-Null Count    Dtype  
---  ------        --------------    -----  
 0   title         1458455 non-null  str    
 1   vote_average  1458474 non-null  float64
 2   vote_count    1458474 non-null  int64  
 3   release_date  1124082 non-null  str    
 4   runtime       1458474 non-null  int64  
 5   adult         1458474 non-null  bool   
 6   overview      1121159 non-null  str    
 7   popularity    1458474 non-null  float64
 8   genres        811136 non-null   str    
 9   keywords      356981 non-null   str    
dtypes: bool(1), float64(2), int64(2), str(5)
memory usage: 448.8 MB
None
             title  vote_average  vote_count release_date  runtime  adult  \
0        Inception         8.364       34495   2010-07-15      148  False   
1     Interstellar         8.417       32571   2014-11-05      169  False   
2  The Dark Knight         8.512       30619   2008-0

In [3]:
print(movies["vote_count"].describe())
print(movies["vote_count"].quantile([0.5, 0.75, 0.9, 0.95, 0.99]))
print("Rows with overview:", movies["overview"].notna().sum())
print("Rows with genres:", movies["genres"].notna().sum())

count    1.458474e+06
mean     1.471322e+01
std      2.812893e+02
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      3.449500e+04
Name: vote_count, dtype: float64
0.50      0.0
0.75      0.0
0.90      4.0
0.95     11.0
0.99    143.0
Name: vote_count, dtype: float64
Rows with overview: 1121159
Rows with genres: 811136


In [4]:
for threshold in [10, 20, 30]:
    subset = movies[
        (movies["vote_count"] >= threshold) &
        movies["overview"].notna() &
        movies["genres"].notna() &
        movies["release_date"].notna() &
        (movies["overview"].str.strip() != "")
    ]
    print(f"vote_count >= {threshold} (with release_date required): {len(subset)} rows")

vote_count >= 10 (with release_date required): 76845 rows
vote_count >= 20 (with release_date required): 49762 rows
vote_count >= 30 (with release_date required): 38419 rows


In [5]:
MIN_VOTE_COUNT = 20

movies = movies.dropna(subset=["title", "overview", "genres", "release_date"])
movies = movies[movies["overview"].str.strip() != ""]

movies = movies[movies["vote_count"] >= MIN_VOTE_COUNT]
movies = movies[movies["adult"] == False]
movies = movies[movies["runtime"] > 0]

movies["keywords"] = movies["keywords"].fillna("")

movies["release_date"] = pd.to_datetime(movies["release_date"], errors="coerce")
movies["year"] = movies["release_date"].dt.year

print(f"Final row count: {len(movies)}")

Final row count: 49503


In [6]:
final_df = movies[[
    "title", "year", "vote_average", "vote_count", "popularity",
    "overview", "genres", "keywords"
]].rename(columns={
    "title": "Title",
    "year": "Year",
    "vote_average": "TMDb_Rating",
    "overview": "Overview",
    "genres": "Genres",
    "keywords": "Keywords"
})

OUTPUT_PATH = "top50k_movies.csv"
final_df.to_csv(OUTPUT_PATH, index=False)

print(f"Final dataset size: {len(final_df)} rows")
print(f"File size: {os.path.getsize(OUTPUT_PATH) / 1e6:.1f} MB")

Final dataset size: 49503 rows
File size: 20.3 MB
